<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_6_integrador_estilo_llm.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_6_integrador_estilo_llm.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F06_foundation_models%2Fnotebooks%2Fsesion_06_6_integrador_estilo_llm.ipynb)

# SI7011 · Sesión 06 · Ejercicio integrador: ¿hay que ajustar un LLM para que responda con un estilo?
**Juan David Martínez Vargas · Universidad EAFIT**

En el notebook 6.3 convertiste un modelo **base** en uno que responde. Aquí partimos de un modelo que **ya conversa**, Qwen2.5-0.5B-Instruct, y queremos que responda en el castellano de *Don Quijote*. La pregunta: **¿basta con pedírselo en el prompt, o hay que ajustarlo?** Lo haces como en la práctica: datos en el formato de chat de TRL, `SFTTrainer` con LoRA de peft, y MLflow.

| Etapa | Qué haces | Artefacto |
|---|---|---|
| Datos | unas 380 preguntas de Dolly con su respuesta reescrita en el estilo: 40 para validar, unas 60 de test y el resto para entrenar (listo) | ficha de datos en MLflow |
| Formato | cada fila como conversación *prompt / completion* (**TODO 1**) | |
| Ajuste | `LoraConfig` y `SFTConfig` (**TODO 2**); `SFTTrainer` registra en MLflow (listo) | `lora_estilo/` |
| Evaluación | $J$ de test para base, base con el estilo en el prompt y LoRA (**TODO 3**); un juez simple del estilo (listo) | tablas en MLflow |
| Despliegue | `responder_raw`: de una pregunta en texto a una respuesta, con contrato (**TODO 4**) | |

**Usa GPU** (Kaggle T4, unos 10 minutos) con **Internet** activado (el modelo viene de Hugging Face y TRL se instala con `pip`). **Datos:** en Kaggle agrega el dataset [`juanmartinezv4399/si7011-s06-dolly-quijote`](https://www.kaggle.com/datasets/juanmartinezv4399/si7011-s06-dolly-quijote) con **Add Input**; en Colab se baja con tu secreto `KAGGLE_API_TOKEN`.

<span style="font-size:smaller">Adaptado de MIT 6.S191 *Introduction to Deep Learning*, Lab 3 (Copyright (c) 2025 MIT Introduction to Deep Learning, licencia MIT). Las preguntas son de Databricks Dolly-15k en español (CC BY-SA 3.0); las respuestas con estilo se generaron con Qwen2.5-7B-Instruct.</span>

In [1]:
%pip install -q mlflow "trl==1.16.0" peft

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')     # Kaggle "GPU T4 x2": una sola GPU (SFTTrainer no admite DataParallel)
# Kaggle trae torchao 0.10, que las versiones recientes de transformers y peft rechazan al importarse
# ("Found an incompatible version of torchao"). Este notebook no lo usa: si es viejo, se desinstala.
import os
from importlib.metadata import version, PackageNotFoundError
try:
    if tuple(int(x) for x in version('torchao').split('.')[:2]) < (0, 16):
        os.system('pip uninstall -y -q torchao')
except (PackageNotFoundError, ValueError):
    pass

import hashlib, json, math, random, textwrap, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers, trl, peft
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
import mlflow
mlflow.set_tracking_uri('sqlite:///mlflow.db')
os.environ['MLFLOW_TRACKING_URI'] = 'sqlite:///mlflow.db'       # el que usa el Trainer de Hugging Face
os.environ['MLFLOW_EXPERIMENT_NAME'] = 'S06-estilo-llm'
mlflow.set_experiment('S06-estilo-llm')

random.seed(0); torch.manual_seed(0)
dev = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'   # Kaggle T4 · Apple Silicon · CPU
print('transformers', transformers.__version__, '· trl', trl.__version__, '· peft', peft.__version__,
      '· MLflow', mlflow.__version__, '·', dev)

/Users/macbook/orca/workspaces/si7011-Martinez-Jimenez-Saldarriaga/Santiago/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1123 00:06:10.955000 69649 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


2026/11/23 00:06:12 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/23 00:06:12 INFO mlflow.store.db.utils: Updating database tables


2026/11/23 00:06:12 INFO mlflow.tracking.fluent: Experiment with name 'S06-estilo-llm' does not exist. Creating a new experiment.


transformers 5.24.0 · trl 1.16.0 · peft 0.22.0 · MLflow 3.18.0 · mps


## 1. Datos (listo)
Cada fila: una pregunta de Dolly en español, la respuesta original (`response`) y la misma respuesta reescrita en el estilo (`response_estilo`). Son preguntas **distintas** de las del notebook 6.3. Del `train` separamos 40 para validar, y de `test` quitamos las preguntas que también están en `train`; `test` se usa solo en el TODO 3. La última línea mide cuántas respuestas de entrenamiento usan algún giro de la época: lo que el modelo tiene para aprender.

In [3]:
KAGGLE_HANDLE = 'juanmartinezv4399/si7011-s06-dolly-quijote'
ROOTS = [Path('.'), Path('data'), Path('/kaggle/input'), Path('/teamspace/studios/this_studio/data')]

def kaggle_token():
    # Colab: secreto KAGGLE_API_TOKEN. Local o Lightning: variable de entorno o ~/.kaggle/access_token.
    if 'KAGGLE_API_TOKEN' not in os.environ:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
        except Exception:
            pass
    return 'KAGGLE_API_TOKEN' in os.environ or (Path.home() / '.kaggle' / 'access_token').exists()

def buscar(roots, patron='dolly_quijote_es.csv'):
    for r in [r for r in roots if r.exists()]:
        hits = sorted(r.rglob(patron))
        if hits:
            return hits[0]
    return None

ruta = buscar(ROOTS)
if ruta is None and kaggle_token():
    import kagglehub                                        # preinstalado en Colab y Kaggle
    ruta = buscar([Path(kagglehub.dataset_download(KAGGLE_HANDLE))])
assert ruta is not None, f'No encontré dolly_quijote_es.csv: en Kaggle agrega {KAGGLE_HANDLE} con Add Input.'
df = pd.read_csv(ruta)
train_all, test = df[df.split == 'train'].reset_index(drop=True), df[df.split == 'test'].reset_index(drop=True)
test = test[~test.instruction.isin(train_all.instruction)].reset_index(drop=True)   # ninguna pregunta de test se vio al entrenar
val, train = train_all.iloc[:40].reset_index(drop=True), train_all.iloc[40:].reset_index(drop=True)
if dev == 'cpu':
    train, val, test = train.iloc[:40], val.iloc[:12], test.iloc[:12]
data_card = {'fuente': str(ruta), 'sha256': hashlib.sha256(ruta.read_bytes()).hexdigest()[:16],
             'train': len(train), 'val': len(val), 'test': len(test)}
print(data_card)
GIROS = ['vuestra merced', 'agora', 'mesmo', 'asaz', 'os digo', 'pardiez', 'aqueste', 'hube de', 'sabed', 'en verdad']
con_giros = lambda r: sum(g in r.lower() for g in GIROS) >= 1      # un juez muy simple del estilo
print(f"respuestas de entrenamiento con algún giro de la época: {train.response_estilo.apply(con_giros).mean():.0%} "
      f"(Dolly original: {train.response.apply(con_giros).mean():.0%})")
e = train.iloc[0]
print('\n▶', e.instruction, '\n  Dolly:  ', e.response, '\n  estilo: ', e.response_estilo)

{'fuente': 'data/dolly_quijote_es.csv', 'sha256': '7f7508c7505180f6', 'train': 284, 'val': 40, 'test': 59}
respuestas de entrenamiento con algún giro de la época: 100% (Dolly original: 0%)

▶ ¿Cuáles son los elementos típicos utilizados para crear un Haft-sin? 
  Dolly:   Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudin dulce de germen de trigo, frutos secos de oleastro, zumaque, vinagre. 
  estilo:  Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudín dulce de germen de trigo, frutos secos de oleastro, zumaque, y vinagre, son los elementos típicos que se usan para crear un Haft-sin, como os digo agora mismo.


## 2. El modelo (listo)
`mensajes` arma la conversación (con o sin mensaje de sistema) y `responder` aplica la **plantilla de chat del modelo** (`apply_chat_template`) y genera con greedy hasta `<|im_end|>`. Mira la plantilla, y cómo responde el modelo con y sin el estilo en el sistema.

In [4]:
MODELO = 'Qwen/Qwen2.5-0.5B-Instruct'      # la versión que ya conversa del modelo base del notebook 6.3
tok = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).to(dev)
FIN = tok.convert_tokens_to_ids('<|im_end|>')               # el fin de turno de la plantilla de Qwen
ESTILO = ('Responde siempre en el castellano de Don Quijote de la Mancha (principios del siglo XVII): '
          'vuestra merced, agora, mesmo, asaz, os digo; tono cortés y algo grandilocuente.')

def mensajes(pregunta, sistema=None):
    return ([{'role': 'system', 'content': sistema}] if sistema else []) + [{'role': 'user', 'content': pregunta}]

@torch.no_grad()
def responder(m, pregunta, sistema=None, max_nuevos=150):
    x = tok.apply_chat_template(mensajes(pregunta, sistema), add_generation_prompt=True, return_tensors='pt',
                                return_dict=True).to(dev)
    out = m.generate(**x, max_new_tokens=max_nuevos, do_sample=False, eos_token_id=FIN,
                     pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, x['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print(tok.apply_chat_template(mensajes('¿Qué es un átomo?'), add_generation_prompt=True, tokenize=False))
for q in test.instruction[:2]:
    print('▶', q)
    print('  base:          ', textwrap.shorten(responder(modelo, q), 300))
    print('  base + estilo: ', textwrap.shorten(responder(modelo, q, ESTILO), 300))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/290 [00:01<06:34,  1.37s/it]

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 211.03it/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
¿Qué es un átomo?<|im_end|>
<|im_start|>assistant

▶ ¿Qué es la bicicleta de gravilla?


  base:           La bicicleta de gravilla es una forma de bicicleta que se utiliza principalmente en el mundo del deporte y la recreación. Aquí hay algunos puntos clave sobre esta bicicleta: 1. Desarrollo: La bicicleta de gravilla fue desarrollada por el ingeniero italiano Luigi Galvani en 1809. 2. Propósito [...]


  base + estilo:  La bicicleta de gravilla es una bicicleta diseñada para ser usada como un vehículo de transporte por tierra. Es conocida por su diseño simple y eficiente, con una sola rueda trasera que permite una velocidad muy rápida. Esta bicicleta se utiliza principalmente para transportar personas o [...]
▶ Cuéntame un chiste sobre un ave originaria de la Antártida.


  base:           Aquí tienes un chiste sobre una ave originaria de la Antártida: ¿Qué hace una ave en la Antártida? Una ave dice: "¡Voy a ver si puedo encontrar un par de huevos!" La respuesta: "¡No te lo pido, no voy a tener huevos!" Espero que disfrutes del chiste! ¿Hay algo más que necesites ayuda con?


  base + estilo:  ¿Quién es más rápido: una ave que ha estado en la Antártida o una que ha estado en la Tierra? La respuesta es: una ave que ha estado en la Antártida.


## 3. TODO 1 · Cada fila como conversación · 5 min
TRL entrena con conversaciones en el formato *prompt / completion*: el prompt es lo que el modelo lee, la completion lo que tiene que aprender a escribir. Escribe `a_conversacion(fila, sistema=None)` que devuelva

```python
{'prompt': mensajes(pregunta, sistema),
 'completion': [{'role': 'assistant', 'content': <la respuesta CON estilo>}]}
```

`conjunto` (listo) convierte una tabla entera en un `Dataset` de Hugging Face.

In [5]:
def a_conversacion(fila, sistema=None):
    """Una fila de la tabla -> una conversación en el formato prompt / completion de TRL.

    - prompt: lo que el modelo LEE. `mensajes` arma la lista de mensajes de chat:
      [sistema (solo si se pasa)] + [usuario con la pregunta de Dolly].
    - completion: lo que el modelo tiene que aprender a ESCRIBIR. Un único mensaje del
      asistente con la respuesta reescrita en el estilo (`response_estilo`), nunca la
      respuesta original de Dolly (`response`).

    Se devuelven listas de mensajes, no texto: SFTTrainer (TRL 1.16.0) aplica él mismo la
    plantilla de chat de Qwen (`apply_chat_template`) y, con `completion_only_loss=True`,
    sabe qué tokens son de la completion para contar la pérdida solo ahí. Por eso aquí no
    se concatena nada ni se agregan etiquetas como <|im_start|> a mano.
    """
    # TODO 1
    return {
        'prompt': mensajes(fila['instruction'], sistema),
        'completion': [{'role': 'assistant', 'content': fila['response_estilo']}],
    }

In [6]:
def conjunto(filas, sistema=None):
    """Un Dataset de Hugging Face en el formato prompt / completion conversacional de TRL."""
    return Dataset.from_list([a_conversacion(f, sistema) for f in filas.to_dict('records')])

ej = a_conversacion(train.iloc[0].to_dict())
assert set(ej) == {'prompt', 'completion'}, "Esperaba las claves 'prompt' y 'completion'."
assert ej['prompt'][-1] == {'role': 'user', 'content': train.instruction[0]}, 'El prompt termina con el mensaje del usuario.'
assert ej['completion'] == [{'role': 'assistant', 'content': train.response_estilo[0]}], \
    'La completion es un solo mensaje del asistente con la respuesta CON estilo.'
ej_s = a_conversacion(train.iloc[0].to_dict(), ESTILO)
assert ej_s['prompt'][0] == {'role': 'system', 'content': ESTILO} and len(ej_s['prompt']) == 2, \
    'Con sistema, el prompt empieza con el mensaje de sistema.'
D_train, D_val = conjunto(train), conjunto(val)
print('a_conversacion correcta ·', len(D_train), 'conversaciones de entrenamiento')
print(D_train[0])

a_conversacion correcta · 284 conversaciones de entrenamiento
{'prompt': [{'role': 'user', 'content': '¿Cuáles son los elementos típicos utilizados para crear un Haft-sin?'}], 'completion': [{'role': 'assistant', 'content': 'Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudín dulce de germen de trigo, frutos secos de oleastro, zumaque, y vinagre, son los elementos típicos que se usan para crear un Haft-sin, como os digo agora mismo.'}]}


## 4. TODO 2 · LoRA y el entrenamiento supervisado · 10 min
Define dos objetos (mira el notebook 6.3, sección 5):
- `lora = LoraConfig(...)`: $r = 16$, `lora_alpha=32`, `lora_dropout=0.0`, `task_type='CAUSAL_LM'`, sobre `q_proj`, `k_proj`, `v_proj`, `o_proj`;
- `args = SFTConfig(...)`: `output_dir='sft_estilo'`, `run_name='1-lora-r16'`, la pérdida **solo en la respuesta** (`completion_only_loss=True`), `max_length=384`, dos épocas, lotes de 8 (entrenamiento y evaluación), `learning_rate=5e-4`, coseno con `warmup_steps=0.1`, `fp16` en la GPU y `bf16=False` (la T4 no tiene bfloat16), evaluación cada 10 pasos (`eval_strategy='steps'`, `eval_steps=10`), `logging_steps=5`, `save_strategy='no'` y `report_to='mlflow'`.

La celda de prueba crea el `SFTTrainer` y te muestra qué tokens cuentan en la pérdida: deben ser solo la respuesta y el fin de turno.

**Predice:** en el TODO 3 vas a medir $J$ sobre las respuestas con estilo de test en tres condiciones. ¿La del modelo con el estilo **pedido en el prompt** quedará más cerca de la del modelo base o de la de LoRA?

**Mi predicción (escrita antes de correr el TODO 3):** la $J_{\text{test}}$ de *base + estilo en el prompt* quedará **mucho más cerca de la de `base` que de la de `LoRA`**. Pedir el estilo en el sistema debería bajar un poco $J$ (el modelo ya ve las palabras *vuestra merced, agora, mesmo...* en el contexto y le da algo más de probabilidad a ese registro), pero un modelo de 0.5B tiene poca capacidad para sostener un castellano del siglo XVII solo por instrucciones: en las dos muestras de la sección 2, *base + estilo* responde casi igual que *base*, sin giros de la época. LoRA, en cambio, ajusta los pesos con 284 respuestas en ese registro, así que espero que baje $J$ con claridad. En números: espero que el prompt cubra **menos de un 25 %** de la distancia entre `base` y `LoRA`.


In [7]:
# TODO 2: LoRA y el entrenamiento supervisado

# LoRA de rango 16 sobre las cuatro proyecciones de la atención; el resto del modelo queda congelado.
lora = LoraConfig(
    r=16,
    lora_alpha=32,                       # escala alpha / r = 2
    lora_dropout=0.0,
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

args = SFTConfig(
    output_dir='sft_estilo',
    run_name='1-lora-r16',               # nombre de la corrida en MLflow
    completion_only_loss=True,           # la pérdida cuenta solo la respuesta (y el fin de turno), no el prompt
    max_length=384,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-4,
    lr_scheduler_type='cosine',
    # Valor literal del enunciado. En transformers >= 5, un warmup_steps < 1 se interpreta como fracción
    # de los pasos de entrenamiento (10 %). En transformers 4.x solo acepta enteros y lanza ValueError.
    warmup_steps=0.1,
    fp16=dev == 'cuda',                  # precisión mixta fp16 solo en la GPU
    bf16=False,                          # la T4 no tiene bfloat16
    eval_strategy='steps',
    eval_steps=10,
    logging_steps=5,
    save_strategy='no',                  # se guarda solo el adaptador, en la sección 5
    report_to='mlflow',
)

/Users/macbook/orca/workspaces/si7011-Martinez-Jimenez-Saldarriaga/Santiago/.venv/lib/python3.12/site-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


In [8]:
assert lora is not None and args is not None, 'Completa TODO 2: define lora y args.'
assert lora.r == 16 and set(lora.target_modules) == {'q_proj', 'k_proj', 'v_proj', 'o_proj'}, \
    'LoRA con r = 16 en las cuatro matrices de la atención.'
assert args.completion_only_loss is True, 'La pérdida debe contar solo la respuesta (completion_only_loss).'
assert args.num_train_epochs == 2 and args.per_device_train_batch_size == 8 and args.learning_rate == 5e-4
assert args.eval_strategy == 'steps' and 'mlflow' in args.report_to, 'Evalúa cada pocos pasos y registra en MLflow.'
assert not args.bf16, 'La T4 no tiene bfloat16: usa fp16.'
trainer = SFTTrainer(model=modelo, args=args, train_dataset=D_train, eval_dataset=D_val,
                     processing_class=tok, peft_config=lora)
trainer.model.print_trainable_parameters()
lote = next(iter(trainer.get_train_dataloader()))
ids, etiquetas = lote['input_ids'][0], lote['labels'][0]
print('cuenta en la pérdida:', repr(tok.decode(ids[etiquetas != -100])))

Tokenizing train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Tokenizing train dataset:  89%|████████▉ | 254/284 [00:00<00:00, 2503.08 examples/s]

Tokenizing train dataset: 100%|██████████| 284/284 [00:00<00:00, 2109.56 examples/s]

Building labels for train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Building labels for train dataset: 100%|██████████| 284/284 [00:00<00:00, 13076.84 examples/s]

Truncating train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

... (9 outputs left out)

Dropping fully masked examples from eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset: 100%|██████████| 40/40 [00:00<00:00, 14857.61 examples/s]

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359
cuenta en la pérdida: 'Aqueste es el gran diferencial entre Spark y MapReduce: mientras que MapReduce opera y guarda los datos en disco para cada paso, hube de decir que Spark los conserva en memoria para los pasos subsiguientes, lo que le confiere una notable agilidad y rapidez.<|im_end|>\n'



/Users/macbook/orca/workspaces/si7011-Martinez-Jimenez-Saldarriaga/Santiago/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


## 5. Entrenar (listo)
`trainer.train()` registra en MLflow los hiperparámetros, la pérdida de entrenamiento y la de validación. Después guardamos **solo el adaptador** (las matrices $A_r$ y $B$, unos MB) con el **contrato**, y los subimos a la misma corrida.

In [9]:
t0 = time.time()
trainer.train()
print(f'{time.time() - t0:.0f} s')
run_id = mlflow.last_active_run().info.run_id              # la corrida que abrió el Trainer
CONTRATO = {'modelo': MODELO, 'max_tokens_pregunta': 128, 'max_nuevos': 150, 'plantilla': 'chat de Qwen, sin sistema'}
trainer.model.save_pretrained('lora_estilo')               # solo el adaptador: unos MB
json.dump(CONTRATO, open('lora_estilo/contrato.json', 'w'), ensure_ascii=False, indent=1)
with mlflow.start_run(run_id=run_id):
    mlflow.log_params({f'data_{k}': v for k, v in data_card.items()})
    mlflow.log_artifacts('lora_estilo', 'lora_estilo')
print('adaptador y contrato en MLflow, corrida', run_id)

/Users/macbook/orca/workspaces/si7011-Martinez-Jimenez-Saldarriaga/Santiago/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,2.330410,2.367463,2.590268,0.523032,12893.000000
20,2.171114,2.175031,2.188711,0.550238,25677.000000
30,2.186733,2.135357,2.251187,0.558759,37567.000000
40,1.897919,2.108624,2.146943,0.563538,49544.000000
50,1.892918,2.109461,2.006323,0.558887,62739.000000
60,1.866959,2.099154,2.019824,0.563352,74944.000000
70,1.878612,2.096503,2.029207,0.564871,87241.000000
72,1.878612,2.096484,2.029232,0.564871,89634.000000


447 s


adaptador y contrato en MLflow, corrida 89226fbeb54d4c7b929d801fd20adbd7


In [10]:
def nll(datos):
    """La J de la respuesta sobre un Dataset, con el modelo como esté (con o sin el adaptador activo)."""
    ev = SFTTrainer(model=trainer.model, processing_class=tok, train_dataset=datos, eval_dataset=datos,
                    args=SFTConfig(output_dir='eval_tmp', per_device_eval_batch_size=8, max_length=384,
                                   fp16=dev == 'cuda', bf16=False, report_to='none'))
    return ev.evaluate()['eval_loss']

## 6. TODO 3 · ¿Basta el prompt? · 15 min
Escribe `comparar(filas)`: devuelve un diccionario con la $J$ de las respuestas con estilo de `filas` en tres condiciones (diapositiva 74):

| Clave | Adaptador | Mensaje de sistema |
|---|---|---|
| `'base'` | apagado (`with trainer.model.disable_adapter():`) | ninguno |
| `'base + estilo en el prompt'` | apagado | `ESTILO` |
| `'LoRA'` | activo | ninguno |

`nll(conjunto(filas, sistema))` ya calcula $J$ sobre un conjunto. La comparación es justa porque las tres condiciones usan el **mismo tokenizador** y la misma respuesta objetivo: solo cambia el prompt o los pesos.

In [11]:
def comparar(filas):
    # TODO 3: {'base': J, 'base + estilo en el prompt': J, 'LoRA': J} sobre las respuestas con estilo de `filas`
    # Mismo tokenizador y misma respuesta objetivo (response_estilo) en las tres condiciones: solo cambian
    # el mensaje de sistema o los pesos (adaptador apagado / activo).
    with trainer.model.disable_adapter():                      # pesos originales de Qwen2.5-0.5B-Instruct
        j_base = nll(conjunto(filas))                          # sin sistema
        j_prompt = nll(conjunto(filas, ESTILO))                # el estilo pedido en el mensaje de sistema
    j_lora = nll(conjunto(filas))                              # adaptador activo, sin sistema
    return {'base': j_base, 'base + estilo en el prompt': j_prompt, 'LoRA': j_lora}


In [12]:
r = comparar(test.iloc[:8])
assert set(r) == {'base', 'base + estilo en el prompt', 'LoRA'}, 'Esperaba las tres condiciones.'
with trainer.model.disable_adapter():
    assert abs(r['base'] - nll(conjunto(test.iloc[:8]))) < 1e-3, '"base" se mide con el adaptador apagado y sin sistema.'
assert abs(r['LoRA'] - nll(conjunto(test.iloc[:8]))) < 1e-3, '"LoRA" se mide con el adaptador activo y sin sistema.'
assert r['base + estilo en el prompt'] != r['base'], 'La condición con el estilo en el prompt usa sistema=ESTILO.'

tabla = pd.DataFrame([{'condición': k, 'J_test': v, 'perplejidad': math.exp(v)} for k, v in comparar(test).items()])
with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla, 'comparacion_test.json')
print(tabla.round(3).to_string(index=False))

/Users/macbook/orca/workspaces/si7011-Martinez-Jimenez-Saldarriaga/Santiago/.venv/lib/python3.12/site-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing train dataset: 100%|██████████| 8/8 [00:00<00:00, 328.73 examples/s]

Building labels for train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for train dataset: 100%|██████████| 8/8 [00:00<00:00, 3467.80 examples/s]

Truncating train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

... (143 outputs left out)

Dropping fully masked examples from eval dataset: 100%|██████████| 59/59 [00:00<00:00, 49090.25 examples/s]


/Users/macbook/orca/workspaces/si7011-Martinez-Jimenez-Saldarriaga/Santiago/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,1.938565,0,1.829344,0.598241,0.000000


                 condición  J_test  perplejidad
                      base   2.593       13.368
base + estilo en el prompt   2.507       12.264
                      LoRA   1.939        6.949


$J$ dice cuánto se parecen las respuestas de test a las que daría el modelo, pero no se lee fácil. Un **juez muy simple**: la fracción de 20 respuestas generadas que usan al menos un giro de la época (`GIROS`, la misma medida de la sección 1). También el largo, porque el ajuste cambia más que el estilo.

In [13]:
N_GEN = 20 if dev != 'cpu' else 2                  # 20 respuestas en GPU (CUDA o MPS)
m = trainer.model
gen = {'base': [], 'base + estilo en el prompt': [], 'LoRA': []}
for q in test.instruction[:N_GEN]:
    with m.disable_adapter():
        gen['base'].append(responder(m, q)); gen['base + estilo en el prompt'].append(responder(m, q, ESTILO))
    gen['LoRA'].append(responder(m, q))
tabla['respuestas con giros de la época'] = [np.mean([con_giros(r) for r in gen[k]]) for k in tabla['condición']]
tabla['palabras por respuesta'] = [np.mean([len(r.split()) for r in gen[k]]) for k in tabla['condición']]
with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla, 'comparacion_test_con_juez.json')
print(tabla.round(3).to_string(index=False), '\n')
for i in range(min(3, N_GEN)):
    print('▶', test.instruction[i])
    for k in gen:
        print(f'  {k + ":":<28}', textwrap.shorten(gen[k][i], 260))
    print()

                 condición  J_test  perplejidad  respuestas con giros de la época  palabras por respuesta
                      base   2.593       13.368                               0.0                   76.20
base + estilo en el prompt   2.507       12.264                               0.0                   55.90
                      LoRA   1.939        6.949                               0.9                   58.05 

▶ ¿Qué es la bicicleta de gravilla?
  base:                        La bicicleta de gravilla es una forma de bicicleta que se utiliza principalmente en el mundo del deporte y la recreación. Aquí hay algunos puntos clave sobre esta bicicleta: 1. Desarrollo: La bicicleta de gravilla fue desarrollada por el ingeniero [...]
  base + estilo en el prompt:  La bicicleta de gravilla es una bicicleta diseñada para ser usada como un vehículo de transporte por tierra. Es conocida por su diseño simple y eficiente, con una sola rueda trasera que permite una velocidad muy rápida. Es

## 7. Cambia una sola cosa · 10 min
Elige **un** cambio: copia la celda del TODO 2 con otro `run_name`, crea un `SFTTrainer` nuevo (con el modelo recién cargado: `AutoModelForCausalLM.from_pretrained(MODELO, ...)`), entrena y repite `comparar(test)`. Antes de correrlo, escribe tu predicción.
- $r = 4$ o $r = 64$;
- LoRA solo en `q_proj` y `v_proj`;
- cuatro épocas en lugar de dos (mira la curva de validación en MLflow);
- entrenar con la respuesta **original** de Dolly en lugar de la con estilo: ¿qué le pasa a la $J$ del estilo?

_Mi cambio y mi predicción:_ **$r = 4$ (con `lora_alpha = 8`) en lugar de $r = 16$ (con `lora_alpha = 32`).** Es un solo cambio, el rango del adaptador: mantengo la escala `lora_alpha / r = 2` para que la magnitud de la actualización $\frac{\alpha}{r} B A$ no cambie y lo único distinto sea cuántas direcciones puede usar. Todo lo demás queda igual que en el TODO 2: las cuatro proyecciones de la atención (`q_proj`, `k_proj`, `v_proj`, `o_proj`), `lora_dropout=0.0`, dos épocas, lotes de 8, `learning_rate=5e-4`, coseno con `warmup_steps=0.1`, `max_length=384`, la pérdida solo en la respuesta, los mismos datos y un modelo recién cargado. La corrida se llama `2-lora-r4` en MLflow.

**Predicción:** el modelo ya sabe español y ya sigue instrucciones; lo que tiene que aprender es un **registro** (vocabulario y giros), un cambio que debería caber en pocas direcciones. Por eso espero que $r = 4$ entrene unas **4 veces menos parámetros** (≈ 0.54 M frente a 2.16 M) y aun así logre una $J_{\text{test}}$ de LoRA **muy parecida a la de $r = 16$**: igual o apenas mayor (diferencia menor a 0.05 nats), y muy por debajo de *base + estilo en el prompt*. Las condiciones `base` y `base + estilo en el prompt` deben dar lo mismo que con $r = 16$, porque con el adaptador apagado el modelo es el mismo.


In [14]:
# 7 · Cambia una sola cosa: LoRA de rango 4 (lora_alpha = 8, misma escala alpha / r = 2) en lugar de 16.
# nll() lee el global `trainer` y la sección 8 descarga el adaptador del global `run_id`:
# se guardan los de r = 16 y se restauran siempre al final (try / finally).
trainer_r16, run_id_r16 = trainer, run_id

modelo_2 = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).to(dev)   # recién cargado, sin adaptador
lora_2 = LoraConfig(
    r=4,                                 # ÚNICO cambio: rango 4 en lugar de 16
    lora_alpha=8,                        # escala alpha / r = 2, igual que en el TODO 2
    lora_dropout=0.0,
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
args_2 = SFTConfig(
    output_dir='sft_estilo',             # save_strategy='no': no se escribe ningún checkpoint
    run_name='2-lora-r4',
    completion_only_loss=True,
    max_length=384,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-4,
    lr_scheduler_type='cosine',
    warmup_steps=0.1,
    fp16=dev == 'cuda',
    bf16=False,
    eval_strategy='steps',
    eval_steps=10,
    logging_steps=5,
    save_strategy='no',
    report_to='mlflow',
)
trainer_2 = SFTTrainer(model=modelo_2, args=args_2, train_dataset=D_train, eval_dataset=D_val,
                       processing_class=tok, peft_config=lora_2)
trainer_2.model.print_trainable_parameters()
t0 = time.time()
trainer_2.train()
print(f'{time.time() - t0:.0f} s')
run_id_2 = mlflow.last_active_run().info.run_id

try:
    trainer = trainer_2                  # comparar() y nll() usan el global `trainer`
    r4 = comparar(test)
finally:
    trainer, run_id = trainer_r16, run_id_r16
assert trainer is trainer_r16 and run_id == run_id_r16

tabla_r4 = pd.DataFrame([{'condición': k, 'J_test': v, 'perplejidad': math.exp(v)} for k, v in r4.items()])
with mlflow.start_run(run_id=run_id_2):
    mlflow.log_table(tabla_r4, 'comparacion_test.json')
    mlflow.log_params({f'data_{k}': v for k, v in data_card.items()})

def val_final(t):
    return [h['eval_loss'] for h in t.state.log_history if 'eval_loss' in h][-1]

def entrenables(t):
    return sum(p.numel() for p in t.model.parameters() if p.requires_grad)

directo = pd.DataFrame({'J_test r = 16': tabla.set_index('condición')['J_test'], 'J_test r = 4': pd.Series(r4)})
directo['diferencia (r4 - r16)'] = directo['J_test r = 4'] - directo['J_test r = 16']
print('\n2-lora-r4 · corrida', run_id_2)
print(tabla_r4.round(3).to_string(index=False), '\n')
print(directo.round(4).to_string())
print(f'\nparámetros entrenables: r = 16 → {entrenables(trainer_r16):,} · r = 4 → {entrenables(trainer_2):,}')
print(f'J de validación al final: r = 16 → {val_final(trainer_r16):.4f} · r = 4 → {val_final(trainer_2):.4f}')


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/290 [00:00<03:23,  1.42it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 408.81it/s]

/Users/macbook/orca/workspaces/si7011-Martinez-Jimenez-Saldarriaga/Santiago/.venv/lib/python3.12/site-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Tokenizing train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Tokenizing train dataset: 100%|██████████| 284/284 [00:00<00:00, 4605.60 examples/s]

... (72 outputs left out)

Dropping fully masked examples from eval dataset: 100%|██████████| 59/59 [00:00<00:00, 52584.77 examples/s]


/Users/macbook/orca/workspaces/si7011-Martinez-Jimenez-Saldarriaga/Santiago/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,1.980899,0,1.970613,0.589973,0.000000



2-lora-r4 · corrida 591a5f4756a348b58975cbdb2bf890de
                 condición  J_test  perplejidad
                      base   2.593       13.368
base + estilo en el prompt   2.507       12.264
                      LoRA   1.981        7.249 

                            J_test r = 16  J_test r = 4  diferencia (r4 - r16)
base                               2.5928        2.5928                 0.0000
base + estilo en el prompt         2.5066        2.5066                 0.0000
LoRA                               1.9386        1.9809                 0.0423

parámetros entrenables: r = 16 → 2,162,688 · r = 4 → 540,672
J de validación al final: r = 16 → 2.0965 · r = 4 → 2.1326


## 8. TODO 4 · Despliegue: `responder_raw` · 10 min
Escribe `responder_raw(pregunta, carpeta)`: recibe una pregunta en texto y la carpeta del adaptador (la que bajamos de MLflow), y devuelve la respuesta en texto. Usa solo lo que hay en la carpeta:
1. el **contrato** (`contrato.json`): la pregunta debe ser un `str` no vacío y tener como máximo `max_tokens_pregunta` tokens; si no, `ValueError`;
2. el modelo: carga el modelo base que dice el contrato y ponle el adaptador con `PeftModel.from_pretrained(base, carpeta)`. Guárdalo en `_CACHE` para no cargarlo en cada llamada;
3. la plantilla **sin** sistema: `responder(modelo, pregunta, max_nuevos=...)` con el `max_nuevos` del contrato.

In [15]:
_CACHE = {}

def responder_raw(pregunta, carpeta='lora_estilo'):
    # TODO 4: valida el contrato, arma base + adaptador (una vez, en _CACHE) y devuelve la respuesta en texto
    carpeta = Path(carpeta)
    contrato = json.loads((carpeta / 'contrato.json').read_text(encoding='utf-8'))
    # 1. Contrato: un texto no vacío, de a lo sumo max_tokens_pregunta tokens del tokenizador del modelo.
    if not isinstance(pregunta, str) or not pregunta.strip():
        raise ValueError('La pregunta debe ser un texto no vacío.')
    clave = str(carpeta)
    if clave not in _CACHE:
        # 2. El modelo base que dice el contrato + el adaptador de la carpeta, cargados una sola vez.
        tok_local = AutoTokenizer.from_pretrained(contrato['modelo'])
        base = AutoModelForCausalLM.from_pretrained(contrato['modelo'], dtype=torch.float32).to(dev)
        m_peft = PeftModel.from_pretrained(base, clave).to(dev).eval()
        _CACHE[clave] = (m_peft, tok_local)
    m_peft, tok_local = _CACHE[clave]
    n = len(tok_local.encode(pregunta, add_special_tokens=False))
    if n > contrato['max_tokens_pregunta']:
        raise ValueError(f"La pregunta tiene {n} tokens; el contrato admite como máximo "
                         f"{contrato['max_tokens_pregunta']}.")
    # 3. La plantilla de chat sin sistema, greedy, con el max_nuevos del contrato.
    return responder(m_peft, pregunta, max_nuevos=contrato['max_nuevos'])


In [16]:
carpeta = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path='lora_estilo')
for q in test.instruction[:3]:
    a, b = responder_raw(q, carpeta), responder(trainer.model, q)
    assert a == b, 'responder_raw debe dar lo mismo que el modelo entrenado (greedy, sin sistema).'
    print('1. Consistencia ✓ ·', textwrap.shorten(a, 150))
for malo in ['', '   ', None, 'palabra ' * 400]:
    try:
        responder_raw(malo, carpeta); raise AssertionError(f'Debió rechazar {str(malo)[:20]!r}.')
    except ValueError as e:
        print('2. Contrato: rechazada →', e)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/290 [00:01<07:56,  1.65s/it]

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 175.42it/s]

1. Consistencia ✓ · La bicicleta de gravilla, que se llama así porque su principal función es llevarla en el aire, es una máquina que se utiliza para transportar [...]


1. Consistencia ✓ · Aqueste es el chiste que os digo: "¿Qué hube de hacer con una avispada que no sabía cómo caminar? En verdad le dijimos que 'hube de darle un [...]


1. Consistencia ✓ · Aqueste es el hallazgo que os digo: la mejor película de todos los tiempos es "La Tierra en el Cielo", dirigida por Christopher Nolan y [...]
2. Contrato: rechazada → La pregunta debe ser un texto no vacío.
2. Contrato: rechazada → La pregunta debe ser un texto no vacío.
2. Contrato: rechazada → La pregunta debe ser un texto no vacío.
2. Contrato: rechazada → La pregunta tiene 402 tokens; el contrato admite como máximo 128.


## 9. Preguntas
1. ¿Se cumplió tu predicción? Con la tabla del TODO 3: ¿cuánto de la distancia entre "base" y "LoRA" cubre el prompt?
2. Mira las respuestas generadas y la columna del juez. ¿Coincide lo que ves con lo que dice $J$? Da un ejemplo en que no coincidan.
3. ¿Por qué no se puede comparar esta $J$ con la NLL del notebook 6.3 ni con la de otro modelo con otro tokenizador?
4. ¿Qué costó cada opción? El prompt: tokens de más en **cada** llamada. LoRA: el entrenamiento y una carpeta de unos MB. ¿Cuándo conviene cada una?
5. En producción llega una pregunta en inglés. ¿Qué haría tu `responder_raw`? ¿Qué agregarías al contrato?

_Respuestas:_

*(Números de esta ejecución: Apple M5 con `mps`, float32; datos `sha256 = 7f7508c7505180f6`, `train = 284`, `val = 40`, `test = 59`; corridas `1-lora-r16` y `2-lora-r4` en el experimento `S06-estilo-llm` de MLflow.)*

**1. ¿Se cumplió la predicción? ¿Cuánto de la distancia entre "base" y "LoRA" cubre el prompt?**
Sí. La tabla del TODO 3 da $J_{\text{test}}$ = **2.593** (perplejidad **13.37**) para `base`, **2.507** (perplejidad **12.26**) para `base + estilo en el prompt` y **1.939** (perplejidad **6.95**) para `LoRA`. El prompt baja $J$ en $2.5928 - 2.5066 = 0.086$ nats, y la distancia entre base y LoRA es $2.5928 - 1.9386 = 0.654$ nats: el prompt cubre apenas el **13.2 %** de esa distancia, por debajo del 25 % que predije. LoRA casi divide la perplejidad a la mitad (13.37 → 6.95), y el prompt solo la baja de 13.37 a 12.26. Con el adaptador de la sección 7 ($r = 4$, $J_{\text{test}}$ de LoRA = **1.981**, perplejidad **7.25**) el prompt cubre el **14.1 %**: la conclusión no depende del rango. También se cumplió la predicción de la sección 7: $r = 4$ entrena **540 672** parámetros frente a **2 162 688** (4 veces menos, 0.11 % del modelo frente a 0.44 %), y su $J_{\text{test}}$ de LoRA es solo **0.042** nats peor (1.981 frente a 1.939). En validación termina en **2.133** frente a **2.097**, y entrenó en 312 s frente a 447 s. Las filas `base` y `base + estilo` coinciden con cuatro decimales entre las dos corridas (2.5928 y 2.5066), lo que confirma que con el adaptador apagado se mide el mismo modelo. Para este cambio de registro, $r = 4$ alcanza casi todo; $r = 16$ gana poco, aunque gana.

**2. ¿Coincide lo que veo con lo que dice $J$? Un ejemplo en que no coincidan.**
En lo grueso, sí: el juez de giros da **0 %** a `base`, **0 %** a `base + estilo en el prompt` y **90 %** a `LoRA` en las 20 respuestas generadas, el mismo orden que $J$. Pero el detalle no coincide en dos puntos:
- **El prompt:** $J$ dice que pedir el estilo ayuda algo (−0.086 nats), y el juez dice que no cambia nada (0 de 20 respuestas con giros). Lo que sí cambia el prompt es el largo (76.2 → 55.9 palabras por respuesta) y a veces el contenido, para peor: en *«¿Cuál es la mejor película de todos los tiempos?»* contesta *«Lo siento, pero no puedo proporcionar una respuesta sobre películas o temas políticos...»*, sin ningún giro de la época. $J$ mide la probabilidad que el modelo le da a la respuesta con estilo de referencia, no lo que el modelo genera con greedy: el prompt mueve un poco esa probabilidad, pero no lo bastante como para que los giros ganen al generar.
- **LoRA:** en *«¿Qué es la bicicleta de gravilla?»* la respuesta de LoRA (*«...que se llama así porque su principal función es llevarla en el aire, es una máquina que se utiliza para transportar objetos o personas...»*) no tiene ningún giro de `GIROS` (el juez le pone 0), aunque LoRA tiene la menor $J$. Al revés, en la película LoRA saca 1 en el juez (*«Aqueste es el hallazgo que os digo...»*) pero inventa los hechos (*«"La Tierra en el Cielo", dirigida por Christopher Nolan y estrenada en 2013»*). Ni $J$ ni el juez miden si la respuesta es cierta: el ajuste aprendió el registro (y la fórmula de arranque *«Aqueste es el ... que os digo»*), no a saber más. LoRA también acorta las respuestas frente a base (58.05 frente a 76.2 palabras), porque las respuestas de entrenamiento con estilo son más cortas.

**3. ¿Por qué no se puede comparar esta $J$ con la NLL del notebook 6.3 ni con la de otro modelo con otro tokenizador?**
$J$ es una NLL **promedio por token**, y solo sobre los tokens de la completion (`completion_only_loss=True`). El número depende de cómo se parte el texto en tokens: otro tokenizador corta la misma respuesta en otra cantidad de tokens con otro vocabulario, así que "nats por token" mide otra unidad. Un tokenizador que usa piezas más largas tiene menos tokens, y cada uno es más difícil de predecir. La $J$ también depende de **qué texto** se puntúa y de **qué condiciona**: en el 6.3 se midió otro modelo (el *base*, sin ajuste de instrucciones), sobre otras preguntas y sobre las respuestas originales de Dolly, con otra plantilla y otra máscara. Aquí se miden las 59 respuestas reescritas en estilo cervantino de test, con la plantilla de chat de Qwen. Comparar sería mezclar cambios de datos, de objetivo y de unidad. La comparación de esta tabla es justa porque las tres condiciones usan el mismo tokenizador, los mismos 59 textos objetivo y la misma máscara. Para comparar entre tokenizadores habría que normalizar por algo independiente del tokenizador, como bits por carácter o por byte del mismo texto.

**4. ¿Qué costó cada opción? ¿Cuándo conviene cada una?**
- **Prompt:** sin entrenamiento y sin artefactos, pero `ESTILO` son 54 tokens y, como reemplaza el sistema por defecto de Qwen, cada llamada lleva **38 tokens más** de entrada (75 frente a 37 para *«¿Qué es un átomo?»*). Es un costo que se paga en **cada** llamada, en cómputo, en latencia y en ventana de contexto. Aquí además casi no funciona: 13.2 % de la mejora de $J$ y 0 % de respuestas con giros.
- **LoRA:** un entrenamiento único de **447 s** en el M5 (2 épocas, 284 ejemplos; en la T4, unos minutos) y un adaptador de **8.3 MB** (`adapter_model.safetensors`, 8 676 008 bytes) que se guarda en MLflow con el contrato. En inferencia no agrega tokens, y el adaptador se puede fusionar con los pesos para no agregar latencia. A cambio hay que conseguir datos con el estilo (aquí, 284 respuestas reescritas con un modelo de 7B), versionar el adaptador y reentrenar si cambia el estilo o el modelo base.
- **Cuándo conviene cada una:** el prompt conviene para prototipos, estilos que cambian a menudo, pocas llamadas, o modelos grandes que sí siguen bien las instrucciones de estilo. LoRA conviene cuando el estilo es fijo, el volumen de llamadas es alto (los 38 tokens por llamada se acumulan) o el modelo es pequeño y no obedece al prompt, como este de 0.5B. Con $r = 4$ (540 672 parámetros, $J$ = 1.981) se obtiene casi lo mismo que con $r = 16$, con un adaptador 4 veces más pequeño.

**5. En producción llega una pregunta en inglés. ¿Qué haría `responder_raw`? ¿Qué agregaría al contrato?**
`responder_raw` la **aceptaría**: es un `str` no vacío y es corta (*«What is the capital of France?»* son 7 tokens, muy por debajo de los 128 de `max_tokens_pregunta`), así que generaría sin quejarse. El contrato actual no dice nada del idioma. Lo probé aparte con el mismo adaptador `lora_estilo`: respondió **en inglés y sin el estilo** (*«The capital of France is Paris, which is known as "la ville de la mer"... the largest city in Europe by population.»*). Además inventó datos, porque los 284 ejemplos de entrenamiento son todos en español y el adaptador no cubre esa distribución. Al contrato le agregaría:
1. un campo `idioma: "es"` y una validación con un detector de idioma (por ejemplo `langdetect`/`fasttext` con un umbral de confianza) que lance `ValueError` o devuelva una respuesta de política, como «solo atiendo preguntas en español», o que traduzca la pregunta antes;
2. la versión exacta del modelo base (*revision*/hash de Hugging Face) y la del adaptador o el `run_id`, para que la carga sea reproducible;
3. los parámetros de decodificación (`do_sample=False`, `eos = <|im_end|>`) y la plantilla, explícitos;
4. límites de salida y de contenido (largo mínimo y máximo, filtros) y un registro de las preguntas rechazadas, para medir cuánto tráfico queda fuera del dominio.

**Si quieres ir más allá:** el juez de giros es fácil de engañar (una respuesta que solo repite "vuestra merced" saca 1). MIT usa un LLM como juez; ¿qué problemas tendría aquí, y cómo comprobarías que el juez está de acuerdo con una persona?

_Respuesta breve:_ un LLM como juez aquí tendría sesgos: preferiría respuestas largas o parecidas a su propio estilo (las referencias las escribió Qwen2.5-7B), sería sensible al orden y a la redacción del prompt del juez, y un modelo pequeño confunde "arcaico" con "formal". Para validarlo, etiquetaría a mano una muestra (por ejemplo 50 respuestas de las tres condiciones, en orden aleatorio y sin decir de cuál vienen) y mediría el acuerdo juez–persona con kappa de Cohen o con una correlación de rangos, antes de usarlo como métrica.